# CORDEX Future Frequency (Projected Severe Rainfall Events)

This notebook estimates **projected changes in severe rainfall event frequency** using CORDEX daily precipitation output.

**Method continuity with PRISM notebooks:**
- Uses the same fixed extreme-event definition (daily precipitation ≥ 3.0 inches ≈ 76.2 mm)
- Counts event-days (frequency), not intensity
- Compares frequency across defined future epochs (first 30 years vs last 30 years)

**Outputs:**
- Gridded event-frequency counts for each future epoch
- Gridded difference map (late minus early)
- Optional saved tables + a saved map image to `outputs/`


## Interpretation Notes

- This step computes **how often** threshold-exceedance days occur in each epoch.
- The output is an empirical **frequency stress signal** (event-days per 30-year window) derived directly from model grids.
- If you later compare to PRISM epoch totals, remember the scale difference: PRISM outputs are typically aggregated to a place-based unit (e.g., county), while CORDEX is gridded.


In [ ]:
# ----------------------------------
# Repo-aware path setup (REQUIRED)
# ----------------------------------
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

# Resolve repository root (notebooks/ is one level down)
REPO_ROOT = Path("..").resolve()

# Ensure repo root is importable (future-proofing)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Core repo directories
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
DATA_SAMPLE_DIR = DATA_DIR / "sample"

OUTPUTS_DIR = REPO_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
MAPS_DIR = OUTPUTS_DIR / "maps"

DOCS_DIR = REPO_ROOT / "docs"
SAMPLE_DIR = REPO_ROOT / "sample_data"

# Ensure output directories exist
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MAPS_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Raw dir:", RAW_DIR)
print("Tables dir:", TABLES_DIR)
print("Maps dir:", MAPS_DIR)

In [ ]:
# -----------------------------
# USER CONFIGURATION
# -----------------------------

# CORDEX NetCDF filename (place in data/raw/)
# From thesis script example:
#   prec.rcp85.MPI-ESM-LR.WRF.day.NAM-22i.raw.nc
CORDEX_FILENAME = "prec.rcp85.MPI-ESM-LR.WRF.day.NAM-22i.raw.nc"
CORDEX_PATH = RAW_DIR / CORDEX_FILENAME

# If you provide a sample NetCDF later, place it in data/sample/ and set this name.
SAMPLE_CORDEX_FILENAME = "cordex_daily_precip_sample.nc"
SAMPLE_CORDEX_PATH = DATA_SAMPLE_DIR / SAMPLE_CORDEX_FILENAME

# Variable name expected in the NetCDF (thesis uses 'prec')
# If None, the notebook will attempt to auto-detect.
PRECIP_VAR = "prec"

# Extreme rainfall threshold in millimeters (3 inches ≈ 76.2 mm)
EXTREME_THRESHOLD_MM = 76.2

# Spatial subset (North Carolina region slice used in thesis script)
LAT_SLICE = (33.8, 37.0)
LON_SLICE = (-84.7, -75.4)

# Future epoch windows (as used in thesis script)
EARLY_START = "2021-01-01"
EARLY_END   = "2050-01-01"   # end is exclusive in xarray slicing logic
LATE_START  = "2051-01-01"
LATE_END    = "2080-01-01"   # end is exclusive

# Output names
MODEL_LABEL = "rcp85.MPI-ESM-LR.WRF"
OUT_DIFF_NETCDF = TABLES_DIR / "cordex_future_extreme_freq_diff.nc"
OUT_EARLY_NETCDF = TABLES_DIR / "cordex_future_extreme_freq_early.nc"
OUT_LATE_NETCDF = TABLES_DIR / "cordex_future_extreme_freq_late.nc"
OUT_MAP_PNG = MAPS_DIR / "cordex_future_extreme_freq_diff.png"

In [ ]:
# Select input source (real data preferred, sample as fallback)
if CORDEX_PATH.exists():
    INPUT_PATH = CORDEX_PATH
elif SAMPLE_CORDEX_PATH.exists():
    INPUT_PATH = SAMPLE_CORDEX_PATH
    print(f"Using sample CORDEX data from {SAMPLE_CORDEX_PATH}")
else:
    raise FileNotFoundError(
        "No CORDEX NetCDF found.\n"
        "Provide either:\n"
        f"- {CORDEX_PATH} (real CORDEX file)\n"
        f"- {SAMPLE_CORDEX_PATH} (sample file)"
    )

print("Using input:", INPUT_PATH)

In [ ]:
# Load dataset
ds = xr.open_dataset(INPUT_PATH)
print(ds)

# Select precip variable
if PRECIP_VAR is not None and PRECIP_VAR in ds.data_vars:
    precip = ds[PRECIP_VAR]
else:
    # Auto-detect: choose first data_var
    auto_var = list(ds.data_vars.keys())[0]
    print(f"PRECIP_VAR not found; using auto-detected variable: {auto_var}")
    precip = ds[auto_var]

# Basic sanity checks
if "time" not in precip.dims:
    raise ValueError(f"Expected a 'time' dimension; got dims={precip.dims}")
if "lat" not in precip.dims or "lon" not in precip.dims:
    raise ValueError(f"Expected 'lat' and 'lon' dimensions; got dims={precip.dims}")

print("Selected precip variable:", precip.name)
print("Time range:", str(precip["time"].min().values), "to", str(precip["time"].max().values))

In [ ]:
# Subset to region (North Carolina slice used in thesis script)
lat_min, lat_max = LAT_SLICE
lon_min, lon_max = LON_SLICE

# Use slice ordering exactly as provided (xarray handles ascending/descending coordinate arrays)
precip_region = precip.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

print("Region subset dims:", precip_region.dims)
print("Region lat range:", float(precip_region["lat"].min()), "to", float(precip_region["lat"].max()))
print("Region lon range:", float(precip_region["lon"].min()), "to", float(precip_region["lon"].max()))

In [ ]:
# Flag extreme-event days (threshold exceedance)
# This produces a boolean grid for each day.
extreme_flag = precip_region >= EXTREME_THRESHOLD_MM

# Confirm
print("Extreme flag dtype:", extreme_flag.dtype)
print("Extreme threshold (mm):", EXTREME_THRESHOLD_MM)

In [ ]:
# Count extreme-event days in early future window
early_counts = extreme_flag.sel(time=slice(EARLY_START, EARLY_END)).sum(dim="time")
early_counts = early_counts.astype(np.int32)

print("Early window:", EARLY_START, "to", EARLY_END)
print("Early counts stats:")
print("  min:", int(early_counts.min().values))
print("  max:", int(early_counts.max().values))

In [ ]:
# Count extreme-event days in late future window
late_counts = extreme_flag.sel(time=slice(LATE_START, LATE_END)).sum(dim="time")
late_counts = late_counts.astype(np.int32)

print("Late window:", LATE_START, "to", LATE_END)
print("Late counts stats:")
print("  min:", int(late_counts.min().values))
print("  max:", int(late_counts.max().values))

In [ ]:
# Difference: late minus early (projected change in extreme-event-day frequency)
freq_diff = (late_counts - early_counts).astype(np.int32)
freq_diff.name = "extreme_event_day_count_diff"

print("Difference stats (late - early):")
print("  min:", int(freq_diff.min().values))
print("  max:", int(freq_diff.max().values))

In [ ]:
# Save outputs to NetCDF for reproducibility / downstream mapping
early_out = early_counts.rename("extreme_event_day_count_early")
late_out = late_counts.rename("extreme_event_day_count_late")

early_out.to_netcdf(OUT_EARLY_NETCDF)
late_out.to_netcdf(OUT_LATE_NETCDF)
freq_diff.to_netcdf(OUT_DIFF_NETCDF)

print("Saved:")
print(" -", OUT_EARLY_NETCDF)
print(" -", OUT_LATE_NETCDF)
print(" -", OUT_DIFF_NETCDF)

In [ ]:
# Plot map (aligned with thesis plotting approach)
import matplotlib.pyplot as plt

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
except Exception as e:
    raise ImportError(
        "Cartopy is required for mapping in this notebook. "
        "Install it (e.g., conda-forge cartopy) and rerun.\n"
        f"Original error: {e}"
    )

projection = ccrs.Mercator()
data_crs = ccrs.PlateCarree()

plt.figure(figsize=(16, 9), dpi=150)
ax = plt.axes(projection=projection)

# Gridlines (kept simple; no styling assumptions)
gl = ax.gridlines(crs=data_crs, draw_labels=True)
gl.top_labels = False
gl.right_labels = False

# Features
ax.add_feature(cfeature.COASTLINE.with_scale("50m"))
ax.add_feature(cfeature.BORDERS.with_scale("50m"))
ax.add_feature(cfeature.STATES.with_scale("50m"))

# Extent
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=data_crs)

# Plot the difference field
# Keep level count similar to thesis (21 levels)
freq_diff.plot.contourf(
    ax=ax,
    transform=data_crs,
    levels=21,
    cbar_kwargs={
        "orientation": "horizontal",
        "shrink": 0.6,
        "pad": 0.05,
        "aspect": 40,
        "label": "Difference in projected ≥3-inch days (late − early)"
    },
)

plt.title(f"CORDEX projected change in ≥3-inch event-days\nModel: {MODEL_LABEL}")

plt.savefig(OUT_MAP_PNG, bbox_inches="tight")
plt.show()

print("Saved map:", OUT_MAP_PNG)

## Next Steps

This notebook produces gridded future frequency change signals (late minus early) for a fixed extreme-event definition.

These outputs are used in:
- **Notebook 05** (exposure + vulnerability overlays)
- **Notebook 06** (flood probability context)

Users may proceed directly to the next notebook after confirming this step completes successfully.
